# Data Exploration

Explore the Turkish fine-tuning datasets used by LowResource-LLM-Forge.

This notebook loads the configured data sources, runs the preprocessing pipeline,
and visualises key statistics: record counts, text-length distributions,
deduplication impact, and sample quality.

In [ ]:
from __future__ import annotations

import json
import sys
from collections import Counter
from pathlib import Path

# Ensure the src directory is on the path so forge imports work.
ROOT = Path.cwd().parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from forge.utils.config import load_data_config  # noqa: E402

print(f"Project root: {ROOT}")

## 1. Load Configuration

The data pipeline is driven by a YAML config. Let's inspect the Turkish config.

In [ ]:
config_path = ROOT / "configs" / "data" / "turkish.yaml"
config = load_data_config(config_path)

print(f"Language: {config.language_name} ({config.language})")
print(f"Sources: {len(config.sources)}")
for src in config.sources:
    print(f"  - {src.repo} (format={src.format}, split={src.split})")
print("\nPreprocessing:")
print(f"  min_length: {config.preprocessing.min_length}")
print(f"  max_length: {config.preprocessing.max_length}")
method = config.preprocessing.dedup_method
threshold = config.preprocessing.dedup_threshold
print(f"  dedup: {method} (threshold={threshold})")
print("\nOutput paths:")
print(f"  SFT:  {config.output.sft_path}")
print(f"  Eval: {config.output.eval_path}")

## 2. Download a Sample

Download a small slice (first 500 records per source) for local analysis.
This avoids downloading full datasets on a laptop.

In [ ]:
from forge.data.collector import DataCollector  # noqa: E402

collector = DataCollector(config)
raw_paths = collector.collect_all(limit=500)

print(f"\nDownloaded {len(raw_paths)} raw files:")
for p in raw_paths:
    line_count = sum(1 for _ in open(p, encoding="utf-8"))
    print(f"  {p.name}: {line_count} records")

## 3. Raw Data Statistics

Before preprocessing, let's inspect the shape of the raw data.

In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    """Load a JSONL file into a list of dicts."""
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return records


all_raw_records: list[dict] = []
for p in raw_paths:
    all_raw_records.extend(load_jsonl(p))

print(f"Total raw records: {len(all_raw_records)}")

# Text length distribution
lengths = []
for rec in all_raw_records:
    text = " ".join(
        str(rec.get(k, ""))
        for k in ("instruction", "input", "output")
        if rec.get(k)
    )
    lengths.append(len(text))

print("\nText length stats (chars):")
print(f"  Min: {min(lengths) if lengths else 0}")
print(f"  Max: {max(lengths) if lengths else 0}")
if lengths:
    print(f"  Mean: {sum(lengths) / len(lengths):.0f}")
    print(f"  Median: {sorted(lengths)[len(lengths) // 2]}")

In [ ]:
# Visualise length distribution
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(1, 1, figsize=(10, 4))
    ax.hist(lengths, bins=50, edgecolor="black", alpha=0.7, color="#2196F3")
    ax.set_xlabel("Text length (chars)")
    ax.set_ylabel("Count")
    ax.set_title("Raw record text-length distribution")
    min_len = config.preprocessing.min_length
    max_len = config.preprocessing.max_length
    ax.axvline(min_len, color="red", linestyle="--", label=f"min={min_len}")
    ax.axvline(max_len, color="orange", linestyle="--", label=f"max={max_len}")
    ax.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed -- skipping histogram.")

## 4. Preprocessing Pipeline

Run the preprocessing pipeline (cleaning, dedup, length filter) and compare
before/after counts.

In [ ]:
from forge.data.preprocessor import DataPreprocessor  # noqa: E402

preprocessor = DataPreprocessor(config.preprocessing)

all_stats: list[dict[str, int]] = []
processed_paths: list[Path] = []

for raw_path in raw_paths:
    out_name = f"clean_{raw_path.name}"
    processed_path = Path("data/processed") / config.language / out_name
    stats = preprocessor.process_file(raw_path, processed_path)
    all_stats.append(stats)
    processed_paths.append(processed_path)

    print(f"\n{raw_path.name}:")
    for k, v in stats.items():
        print(f"  {k}: {v}")
    if stats["total"] > 0:
        keep_rate = stats["kept"] / stats["total"] * 100
        print(f"  keep_rate: {keep_rate:.1f}%")

In [ ]:
# Aggregate preprocessing summary
total_in = sum(s["total"] for s in all_stats)
total_kept = sum(s["kept"] for s in all_stats)
total_short = sum(s["too_short"] for s in all_stats)
total_long = sum(s["too_long"] for s in all_stats)
total_dup = sum(s["duplicate"] for s in all_stats)

print("=== Preprocessing Summary ===")
print(f"  Total input:  {total_in}")
if total_in:
    print(f"  Kept:         {total_kept} ({total_kept / total_in * 100:.1f}%)")
print(f"  Too short:    {total_short}")
print(f"  Too long:     {total_long}")
print(f"  Duplicates:   {total_dup}")

try:
    import matplotlib.pyplot as plt

    labels = ["Kept", "Too short", "Too long", "Duplicate"]
    sizes = [total_kept, total_short, total_long, total_dup]
    colors = ["#4CAF50", "#FF9800", "#F44336", "#9E9E9E"]
    nonzero = [
        (lab, sz, col)
        for lab, sz, col in zip(labels, sizes, colors, strict=True)
        if sz > 0
    ]

    if nonzero:
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.pie(
            [sz for _, sz, _ in nonzero],
            labels=[lab for lab, _, _ in nonzero],
            colors=[col for _, _, col in nonzero],
            autopct="%1.1f%%",
            startangle=90,
        )
        ax.set_title("Preprocessing Filter Breakdown")
        plt.tight_layout()
        plt.show()
except ImportError:
    pass

## 5. Sample Inspection

Look at a few cleaned records to assess data quality.

In [ ]:
from IPython.display import Markdown, display  # noqa: E402

clean_records: list[dict] = []
for p in processed_paths:
    if p.exists():
        clean_records.extend(load_jsonl(p))

print(f"Total clean records available: {len(clean_records)}\n")

for i, rec in enumerate(clean_records[:5]):
    instruction = rec.get("instruction", "")
    output = rec.get("output", "")
    md = (
        f"---\n**Sample {i + 1}**\n\n"
        f"**Instruction:** {instruction[:200]}\n\n"
        f"**Output:** {output[:300]}"
    )
    display(Markdown(md))

## 6. Dataset Builder: Train / Eval Split

Build the final SFT dataset and verify the train/eval split ratio.

In [ ]:
from forge.data.dataset_builder import DatasetBuilder  # noqa: E402

builder = DatasetBuilder(config)
sft_path, eval_path = builder.build(processed_paths)

sft_count = sum(1 for _ in open(sft_path, encoding="utf-8"))
eval_count = sum(1 for _ in open(eval_path, encoding="utf-8"))

print(f"SFT training set: {sft_count} records ({sft_path})")
print(f"Eval set:         {eval_count} records ({eval_path})")
actual = eval_count / (sft_count + eval_count) * 100
target = config.output.eval_split_ratio * 100
print(f"Eval ratio:       {actual:.1f}%")
print(f"Target ratio:     {target:.1f}%")

## 7. Turkish Language Quality Check

Verify that the cleaned data contains proper Turkish text by checking for
Turkish-specific characters and common words.

In [ ]:
TURKISH_CHARS = set("cCgGiIsSOoUu")
TURKISH_WORDS = {
    "ve", "bir", "bu", "de", "da", "ile", "icin", "olan", "var", "gibi",
}

has_turkish_chars = 0
turkish_word_hits: Counter[str] = Counter()

for rec in clean_records:
    text = " ".join(
        str(rec.get(k, "")) for k in ("instruction", "output")
    )
    if any(c in TURKISH_CHARS for c in text):
        has_turkish_chars += 1
    words = set(text.lower().split())
    for tw in TURKISH_WORDS & words:
        turkish_word_hits[tw] += 1

total = len(clean_records) or 1
pct = has_turkish_chars / total * 100
print(f"Records with Turkish characters: {has_turkish_chars}/{total} ({pct:.1f}%)")
print("\nMost common Turkish words:")
for word, count in turkish_word_hits.most_common(10):
    print(f"  {word}: {count}")

## Summary

This notebook demonstrated:
1. Loading data pipeline config from YAML
2. Downloading a sample from HuggingFace sources
3. Inspecting raw data statistics (lengths, counts)
4. Running the cleaning and deduplication pipeline
5. Building the SFT train/eval split
6. Verifying Turkish language content quality

Next: Run `02_training_analysis.ipynb` after a training run to analyse loss
curves and model performance.